In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/datasets/muhammadsaadmirza1/dataset/validation-00000-of-00001.parquet
/kaggle/input/datasets/muhammadsaadmirza1/dataset/train-00000-of-00001.parquet
/kaggle/input/datasets/muhammadsaadmirza1/dataset/test-00000-of-00001.parquet


In [2]:
import torch
print("CUDA available:", torch.cuda.is_available())
print("GPU name:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "None")

CUDA available: True
GPU name: Tesla T4


In [3]:
import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

/kaggle/input/datasets/muhammadsaadmirza1/dataset/validation-00000-of-00001.parquet
/kaggle/input/datasets/muhammadsaadmirza1/dataset/train-00000-of-00001.parquet
/kaggle/input/datasets/muhammadsaadmirza1/dataset/test-00000-of-00001.parquet


In [4]:
!pip install -U bitsandbytes transformers accelerate peft datasets pyyaml -q

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 42.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.3/12.3 MB 57.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 394.3/394.3 kB 24.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 832.9/832.9 kB 44.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 22.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 516.0/516.0 kB 32.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.4/3.4 MB 89.1 MB/s eta 0:00:00


In [5]:
import os
os.makedirs("/kaggle/working/configs", exist_ok=True)
os.makedirs("/kaggle/working/src", exist_ok=True)
os.makedirs("/kaggle/working/scripts", exist_ok=True)
os.makedirs("/kaggle/working/checkpoints", exist_ok=True)
os.makedirs("/kaggle/working/logs", exist_ok=True)
os.makedirs("/kaggle/working/outputs", exist_ok=True)
print("Folders created")

Folders created


In [6]:
%%writefile /kaggle/working/configs/config.yaml
model_name: "microsoft/phi-2"
project_root: "/kaggle/working"

data:
  train: "/kaggle/input/datasets/muhammadsaadmirza1/dataset/train-00000-of-00001.parquet"
  validation: "/kaggle/input/datasets/muhammadsaadmirza1/dataset/validation-00000-of-00001.parquet"
  test: "/kaggle/input/datasets/muhammadsaadmirza1/dataset/test-00000-of-00001.parquet"
  text_column: "text"
  max_length: 128
  train_sample_size: null

training:
  output_dir: "checkpoints"
  final_model_dir: "outputs/fine_tuned_phi2"
  num_train_epochs: 3
  per_device_train_batch_size: 4
  per_device_eval_batch_size: 2
  gradient_accumulation_steps: 4
  learning_rate: 0.0002
  logging_dir: "logs"
  logging_steps: 20
  eval_strategy: "steps"
  eval_steps: 500
  save_steps: 500
  save_total_limit: 3
  warmup_ratio: 0.03
  early_stopping_patience: 2

lora:
  r: 16
  lora_alpha: 32
  lora_dropout: 0.05
  target_modules: ["q_proj", "k_proj", "v_proj", "dense"]

inference:
  max_new_tokens: 120
  temperature: 0.8
  top_p: 0.9

Writing /kaggle/working/configs/config.yaml


In [7]:
%%writefile /kaggle/working/src/train_phi2.py
import os
import math
import inspect
import yaml
import torch
import numpy as np
import pandas as pd
from datasets import Dataset
from transformers import (
    AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig,
    TrainingArguments, Trainer, DataCollatorForLanguageModeling,
    EarlyStoppingCallback
)
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

with open("configs/config.yaml") as f:
    cfg = yaml.safe_load(f)

os.chdir(cfg["project_root"])

MODEL_NAME = cfg["model_name"]
MAX_LEN = cfg["data"]["max_length"]
TEXT_COL = cfg["data"]["text_column"]

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
    trust_remote_code=True,
)
model = prepare_model_for_kbit_training(model, use_gradient_checkpointing=True)

lora_cfg = LoraConfig(
    r=cfg["lora"]["r"],
    lora_alpha=cfg["lora"]["lora_alpha"],
    lora_dropout=cfg["lora"]["lora_dropout"],
    target_modules=cfg["lora"]["target_modules"],
    task_type="CAUSAL_LM",
)
model = get_peft_model(model, lora_cfg)
model.print_trainable_parameters()


def load_split(path):
    df = pd.read_parquet(path)
    return Dataset.from_pandas(df[[TEXT_COL]])


train_ds = load_split(cfg["data"]["train"])
val_ds = load_split(cfg["data"]["validation"])

sample_size = cfg["data"].get("train_sample_size")
if sample_size and sample_size < len(train_ds):
    train_ds = train_ds.shuffle(seed=42).select(range(sample_size))
    print("Using", sample_size, "sampled rows out of full training set")


def tokenize_fn(batch):
    out = tokenizer(
        batch[TEXT_COL],
        truncation=True,
        max_length=MAX_LEN,
        padding="max_length",
    )
    out["labels"] = out["input_ids"].copy()
    return out


train_ds = train_ds.map(tokenize_fn, batched=True, remove_columns=[TEXT_COL])
val_ds = val_ds.map(tokenize_fn, batched=True, remove_columns=[TEXT_COL])

data_collator = DataCollatorForLanguageModeling(tokenizer=tokenizer, mlm=False)


def preprocess_logits_for_metrics(logits, labels):
    return logits.argmax(dim=-1)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    predictions = predictions[:, :-1]
    labels = labels[:, 1:]
    mask = labels != -100
    correct = (predictions == labels) & mask
    token_accuracy = correct.sum() / mask.sum()
    return {"token_accuracy": float(token_accuracy)}


t = cfg["training"]

valid_params = set(inspect.signature(TrainingArguments.__init__).parameters.keys())

desired_args = {
    "output_dir": t["output_dir"],
    "num_train_epochs": t["num_train_epochs"],
    "per_device_train_batch_size": t["per_device_train_batch_size"],
    "per_device_eval_batch_size": t["per_device_eval_batch_size"],
    "gradient_accumulation_steps": t["gradient_accumulation_steps"],
    "learning_rate": float(t["learning_rate"]),
    "logging_dir": t["logging_dir"],
    "logging_steps": t["logging_steps"],
    "eval_strategy": t["eval_strategy"],
    "eval_steps": t["eval_steps"],
    "save_strategy": "steps",
    "save_steps": t["save_steps"],
    "save_total_limit": t["save_total_limit"],
    "warmup_ratio": t["warmup_ratio"],
    "fp16": True,
    "report_to": "none",
    "load_best_model_at_end": True,
    "metric_for_best_model": "eval_loss",
    "greater_is_better": False,
    "eval_accumulation_steps": 1,
}

if "eval_strategy" not in valid_params and "evaluation_strategy" in valid_params:
    desired_args["evaluation_strategy"] = desired_args.pop("eval_strategy")

filtered_args = {k: v for k, v in desired_args.items() if k in valid_params}
dropped = set(desired_args.keys()) - set(filtered_args.keys())
if dropped:
    print("Note: skipped unsupported TrainingArguments params:", dropped)

training_args = TrainingArguments(**filtered_args)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_ds,
    eval_dataset=val_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    preprocess_logits_for_metrics=preprocess_logits_for_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=t["early_stopping_patience"])],
)

if __name__ == "__main__":
    last_checkpoint = None
    if os.path.isdir(t["output_dir"]):
        checkpoints = [d for d in os.listdir(t["output_dir"]) if d.startswith("checkpoint-")]
        if checkpoints:
            last_checkpoint = os.path.join(
                t["output_dir"],
                sorted(checkpoints, key=lambda x: int(x.split("-")[1]))[-1]
            )
            print("Resuming from checkpoint:", last_checkpoint)

    train_result = trainer.train(resume_from_checkpoint=last_checkpoint)
    eval_metrics = trainer.evaluate()
    eval_metrics["perplexity"] = math.exp(eval_metrics["eval_loss"])

    print("===== FINAL METRICS =====")
    print("Train loss:", round(train_result.training_loss, 4))
    print("Validation loss:", round(eval_metrics["eval_loss"], 4))
    print("Validation perplexity:", round(eval_metrics["perplexity"], 4))
    print("Validation token accuracy:", round(eval_metrics["eval_token_accuracy"], 4))

    model.save_pretrained(t["final_model_dir"])
    tokenizer.save_pretrained(t["final_model_dir"])
    print("Saved fine-tuned model to", t["final_model_dir"])

Writing /kaggle/working/src/train_phi2.py


In [8]:
!python src/train_phi2.py

config.json: 100%|█████████████████████████████| 735/735 [00:00<00:00, 3.32MB/s]
tokenizer_config.json: 7.34kB [00:00, 29.6MB/s]
vocab.json: 798kB [00:00, 21.0MB/s]
merges.txt: 456kB [00:00, 89.5MB/s]
added_tokens.json: 1.08kB [00:00, 6.09MB/s]
special_tokens_map.json: 100%|████████████████| 99.0/99.0 [00:00<00:00, 882kB/s]
tokenizer.json: 2.11MB [00:00, 144MB/s]
model.safetensors.index.json: 35.7kB [00:00, 24.8MB/s]
Fetching 2 files: 100%|███████████████████████████| 2/2 [00:45<00:00, 22.79s/it]
Download complete: 100%|████████████████████| 5.56G/5.56G [00:45<00:00, 122MB/s]
generation_config.json: 100%|███████████████████| 124/124 [00:00<00:00, 498kB/s]
trainable params: 10,485,760 || all params: 2,790,169,600 || trainable%: 0.3758
Map: 100%|█████████████████████████| 7852/7852 [00:00<00:00, 8864.01 examples/s]
Note: skipped unsupported TrainingArguments params: {'logging_dir', 'warmup_ratio'}
  0%|                                                 | 0/13254 [00:00<?, ?it/s]/usr/local/